# Train the Skin-Check AI model 🧠

This notebook trains the neural network for the **Cancer Risk-Factor Checker** website (version 2).

**What it does:** downloads the public **HAM10000** dermatology dataset (10,015 real photos of skin spots),
fine-tunes a MobileNetV2 network to tell *benign-looking* from *suspicious-looking* spots, measures how
accurate it is on photos it never saw during training, and exports the model for the website.

**Before you press Run:**
1. You need a free [Kaggle](https://www.kaggle.com) account (the dataset is hosted there).
2. In the menu: **Runtime → Change runtime type → T4 GPU** (free). Training takes ~20-30 minutes.
3. If you already ran an older version of this notebook in this session, do
   **Runtime → Restart session** first, so the Keras switch in the first cell can take effect.
4. Then: **Runtime → Run all**. When a cell asks you to log in to Kaggle, follow the link it shows.

If the first cell's last line fails with "Keras 3 detected", just do
**Runtime → Restart session and run all** once — that always fixes it.

In [ ]:
# Switch TensorFlow to classic Keras 2 — required for the website export step.
# This must happen BEFORE TensorFlow is imported, which is why it's the first cell.
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

!pip -q install tensorflowjs tf_keras kagglehub

import tensorflow as tf
print('TensorFlow', tf.__version__, '| Keras:', tf.keras.__version__,
      '| GPU:', tf.config.list_physical_devices('GPU'))
assert tf.keras.__version__.startswith('2.'), 'Keras 3 detected — do Runtime > Restart session and run all'

In [ ]:
# Download the HAM10000 dataset from Kaggle (about 3 GB).
import kagglehub, glob, os

data_dir = kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000')
print('Downloaded to:', data_dir)

# Map every image id to its file path (images are split across two folders).
image_paths = {os.path.splitext(os.path.basename(p))[0]: p
               for p in glob.glob(os.path.join(data_dir, '**', '*.jpg'), recursive=True)}
print('Found', len(image_paths), 'images')

In [ ]:
# Build the labels. HAM10000 has 7 diagnosis types; we group them into two classes.
# Malignant / needs-attention: melanoma (mel), basal cell carcinoma (bcc), actinic keratoses (akiec)
# Benign: moles (nv), benign keratoses (bkl), dermatofibroma (df), vascular lesions (vasc)
import pandas as pd
from sklearn.model_selection import train_test_split

meta_csv = glob.glob(os.path.join(data_dir, '**', 'HAM10000_metadata*'), recursive=True)[0]
df = pd.read_csv(meta_csv)
df['path'] = df['image_id'].map(image_paths)
df = df.dropna(subset=['path'])
df['label'] = df['dx'].isin(['mel', 'bcc', 'akiec']).astype(int)
print(df['label'].value_counts().rename({0: 'benign', 1: 'suspicious'}))

# 70% train, 15% validation, 15% test — stratified so class balance is kept.
train_df, rest_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(rest_df, test_size=0.50, stratify=rest_df['label'], random_state=42)
print(len(train_df), 'train /', len(val_df), 'validation /', len(test_df), 'test images')

In [ ]:
# Turn the file lists into fast TensorFlow data pipelines.
IMG = 224
BATCH = 32

def load_image(path, label):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.image.resize(img, [IMG, IMG])
    return img, label  # raw 0-255 pixels; the model itself rescales them

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

def make_ds(frame, training=False):
    ds = tf.data.Dataset.from_tensor_slices((frame['path'].values, frame['label'].values))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(2000).map(lambda x, y: (augment(x, training=True), y),
                                  num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(train_df, training=True)
val_ds = make_ds(val_df)
test_ds = make_ds(test_df)

In [ ]:
# Build the model: MobileNetV2 pretrained on ImageNet, with a new head for our 2 classes.
# The Rescaling layer is INSIDE the model, so the website can feed raw 0-255 pixels.
base = tf.keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet')
base.trainable = False

inputs = tf.keras.Input(shape=(IMG, IMG, 3))
x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1)(inputs)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
model = tf.keras.Model(inputs, outputs)

# The dataset has far more benign images, so weight the rare class higher.
n_benign = (train_df['label'] == 0).sum()
n_mal = (train_df['label'] == 1).sum()
class_weight = {0: len(train_df) / (2 * n_benign), 1: len(train_df) / (2 * n_mal)}
print('Class weights:', class_weight)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
history = model.fit(train_ds, validation_data=val_ds, epochs=5, class_weight=class_weight)

In [ ]:
# Fine-tuning: unfreeze the top of MobileNetV2 and train a little more with a tiny learning rate.
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
history_ft = model.fit(train_ds, validation_data=val_ds, epochs=3, class_weight=class_weight)

In [ ]:
# The honest part: how good is it on 1,500 images it has NEVER seen?
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

loss, test_acc, test_auc = model.evaluate(test_ds)

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_prob = model.predict(test_ds).ravel()
y_pred = (y_prob >= 0.5).astype(int)

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)  # of truly suspicious spots, how many did we catch?
specificity = tn / (tn + fp)  # of truly benign spots, how many did we correctly clear?

print(f'Test accuracy:  {test_acc:.1%}')
print(f'Test AUC:       {test_auc:.3f}')
print(f'Sensitivity:    {sensitivity:.1%}  (suspicious spots correctly flagged)')
print(f'Specificity:    {specificity:.1%}  (benign spots correctly cleared)')

ConfusionMatrixDisplay(cm, display_labels=['benign', 'suspicious']).plot(cmap='Blues')
plt.title('Test-set confusion matrix')
plt.show()

In [ ]:
# Export the model for the website (TensorFlow.js format) and download it.
import tensorflowjs as tfjs
import json, shutil, datetime
from google.colab import files

tfjs.converters.save_keras_model(model, 'tfjs_model')

metrics = {
    'test_accuracy': round(float(test_acc), 4),
    'test_auc': round(float(test_auc), 4),
    'sensitivity': round(float(sensitivity), 4),
    'specificity': round(float(specificity), 4),
    'test_images': int(len(test_df)),
    'trained_on': 'HAM10000, ' + datetime.date.today().isoformat(),
}
with open('tfjs_model/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

shutil.make_archive('skin_model', 'zip', 'tfjs_model')
files.download('skin_model.zip')

## Done! Now put the model on your website 🎉

1. Unzip `skin_model.zip` — you'll get `model.json`, one or more `group1-shard...bin` files, and `metrics.json`.
2. In your website repo, put **all of those files** into the folder `static/model/`.
3. Commit and push to GitHub — Render redeploys, and the AI Skin Check page starts working, showing your real test accuracy.

**For your write-up:** report accuracy, sensitivity and specificity (from the cell above), say the model
was trained on HAM10000 (Tschandl et al., 2018), and be clear that it compares photos to training data —
it does not diagnose anyone.